# MAF Replication on the MemeDecode Dataset — Kaggle GPU run

Runs the **same code** as the local `Replication/Scripts/` folder, on a Kaggle GPU. This is the Kaggle counterpart of `MAF_Replication_Colab.ipynb` — same stages, same defaults, different mounting rules.

Training MAF fine-tunes all ~110M parameters of Bangla-BERT (CLIP stays frozen). On CPU that is a multi-day run; on a Kaggle T4 the paper's configuration (batch 4, 20 epochs) finishes in roughly 1–2 hours.

### Before you start

1. Upload `Replication.zip` (built by `Scripts/make_colab_zip.ps1`) as a **Kaggle Dataset**: kaggle.com/datasets → New Dataset → drop the zip → Create. Kaggle extracts it automatically.
2. Open this notebook via **File → Import Notebook**, or start a fresh notebook from the dataset page (which auto-attaches it).
3. If the dataset isn't already attached, use **"+ Add Input"** in the right panel to attach it.
4. **Settings (right panel) → Accelerator → GPU T4 x2** (or P100). Kaggle's free GPU quota is roughly 30 hrs/week, with a ~9–12 hr session limit.

Kaggle mounts your dataset **read-only** at `/kaggle/input/<slug>/...`. Only `/kaggle/working` is writable, so this notebook copies the (small) `Scripts/` folder there and reads the (large) `Dataset/` folder straight from the read-only mount — no need to duplicate 365 MB of images.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Notebook settings (right panel) -> Accelerator -> GPU T4 x2'

## 2. Locate the attached dataset

Auto-detects the mounted dataset by looking for `Dataset/Img` under `/kaggle/input`, so you don't need to hardcode your dataset's slug.

In [ ]:
import glob, os

candidates = glob.glob('/kaggle/input/*/Dataset/Img') + glob.glob('/kaggle/input/*/*/Dataset/Img')
assert candidates, (
    'No Dataset/Img found under /kaggle/input. '
    'Attach the Replication dataset via "+ Add Input" in the right panel.'
)
INPUT_ROOT = os.path.dirname(os.path.dirname(candidates[0]))  # .../<slug>[/Replication]
print('Dataset mount:', INPUT_ROOT)
print('memes  :', len(os.listdir(f'{INPUT_ROOT}/Dataset/Img')))
print(sorted(os.listdir(INPUT_ROOT)))

## 3. Install dependencies

Kaggle notebooks already ship a CUDA build of torch, so only the extras are installed — notably CLIP from source, exactly as the original `requirements.txt` specifies.

In [ ]:
!pip install -q transformers sentencepiece imbalanced-learn madgrad ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git

import clip, transformers
print('transformers', transformers.__version__)
print('clip OK')

## 4. Set up a writable working copy

`/kaggle/input` is read-only, but `main.py` needs to create `Saved_Models/`, `Outputs/` and a `.cache/` next to `Scripts/`. So `Scripts/` (a few hundred KB) is copied to `/kaggle/working`; `Dataset/` (365 MB of images) stays on the read-only input mount and is referenced by an absolute path — no need to duplicate it.

In [ ]:
import shutil

ROOT = '/kaggle/working/Replication'
os.makedirs(ROOT, exist_ok=True)

if os.path.isdir(f'{ROOT}/Scripts'):
    shutil.rmtree(f'{ROOT}/Scripts')
shutil.copytree(f'{INPUT_ROOT}/Scripts', f'{ROOT}/Scripts')
shutil.copy(f'{INPUT_ROOT}/requirements.txt', ROOT)

DATASET_PATH = f'{INPUT_ROOT}/Dataset'  # absolute path -> main.py reads straight from the input mount
print('Working scripts at:', f'{ROOT}/Scripts')
print('Reading dataset from (read-only):', DATASET_PATH)

## 5. Splits summary (cf. paper Table 1)

In [ ]:
import pandas as pd
for name in ['training_set', 'validation_set', 'testing_set']:
    df = pd.read_csv(f'{DATASET_PATH}/{name}.csv')
    print(f'{name:<16} {len(df):>5} rows   ', dict(df["Label"].value_counts()))

## 6. Smoke test

Runs the full pipeline on a handful of memes for one epoch. This is **not a result** — it only proves the data, the model and the metrics all wire up before committing to a long run.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --dataset "{DATASET_PATH}" --subset 24 --n_iter 1 --run_name kaggle_smoketest

## 7. The real run

Paper hyperparameters (Appendix A): batch 4, 20 epochs, lr 5e-5, 16 attention heads, max_len 70.

The best-validation-accuracy checkpoint is kept and used for the test evaluation, as in the original. Mind Kaggle's session time limit (~9–12 hrs) — this run is expected to take 1–2 hrs on a T4.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --dataset "{DATASET_PATH}" --run_name maf_full --batch_size 4 --n_iter 20 --lrate 5e-5 --heads 16 --max_len 70

## 8. Ablations and variants (optional)

- `--attn_variant paper` — the attention operand order the **paper text** describes (Q from text, K/V from vision), rather than the order the **released code** implements. See README §5.4.
- `--fix_scheduler` — steps the LR scheduler per batch instead of per epoch, correcting the original's scheduler bug. See README §5.5.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --dataset "{DATASET_PATH}" --run_name maf_paper_attn --attn_variant paper
!python main.py --dataset "{DATASET_PATH}" --run_name maf_fixed_sched --fix_scheduler

## 9. Results

Compares every run in `Outputs/` against the paper's published MAF row.

**These numbers are not directly comparable to the paper's** — our task is 4-way rather than 5-way, and our captions are raw OCR (optionally denoised) with no manual correction pass. See README §5.

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{ROOT}/Outputs/results_*.json')):
    r = json.load(open(path, encoding='utf-8'))
    rows.append({
        'run': r['run_name'],
        'Acc': round(r['accuracy'], 3),
        'WF1': round(r['weighted_f1'], 3),
        'MacroF1': round(r['macro_f1'], 3),
        'MMAE': round(r['mmae'], 3),
    })
rows.append({'run': 'PAPER MAF (5-way MIMOSA)', 'Acc': 0.741, 'WF1': 0.742, 'MacroF1': None, 'MMAE': 0.645})
print(pd.DataFrame(rows).to_string(index=False))

## 10. Confusion matrix

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

RUN = 'maf_full'
r = json.load(open(f'{ROOT}/Outputs/results_{RUN}.json', encoding='utf-8'))

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=r['target_names'], yticklabels=r['target_names'], cbar=False)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'MAF — {RUN}')
plt.tight_layout(); plt.show()

## 11. Results persist automatically

Everything under `/kaggle/working` (including `{ROOT}/Outputs` and `{ROOT}/Saved_Models`) is kept as the notebook's **Output** when you commit/save the session — download it from the notebook's "Output" tab afterward. No extra copy step needed (unlike Colab, which needs an explicit copy back to Drive).

In [ ]:
!ls -lh {ROOT}/Outputs {ROOT}/Saved_Models

## 12. Generate the Kaggle submission

Produces `submission.csv` from the trained checkpoint, in the exact format
`sample_submission.csv` requires (`Image_name,Target`, using the original
`Neutral/Genders/Politics/Religion` vocabulary). This script never touches ground-truth
labels - it only reads images + captions and writes out what the model predicts.

If `sample_submission.csv` was bundled into the attached dataset it is picked up
automatically; otherwise attach it as a second Kaggle Dataset input and adjust the path.

Each run now saves its own checkpoint, `Saved_Models/maf_model_<run_name>.pth` — earlier
versions of this notebook wrote every run to one shared `maf_model.pth`, so later runs
overwrote earlier models. The cell below submits `maf_full`; change the name to submit
another run.

The cell after it needs **no checkpoint at all**: every `predictions_<run>.csv` already
holds that run's prediction for all 400 submission images, so
`predictions_to_submission.py` turns each one into `submission_<run>.csv`.

In [ ]:
import glob

sub_candidates = glob.glob('/kaggle/input/*/sample_submission.csv') + glob.glob('/kaggle/input/*/*/sample_submission.csv')
SAMPLE_SUB = sub_candidates[0] if sub_candidates else f'{INPUT_ROOT}/sample_submission.csv'
print('Using sample_submission.csv at:', SAMPLE_SUB)

%cd {ROOT}/Scripts
!python generate_submission.py --checkpoint {ROOT}/Saved_Models/maf_model_maf_full.pth --sample_submission "{SAMPLE_SUB}" --out {ROOT}/Outputs/submission.csv

import pandas as pd
pd.read_csv(f'{ROOT}/Outputs/submission.csv').head()

In [ ]:
%cd {ROOT}/Scripts
# One submission_<run>.csv per finished run, straight from its predictions CSV (no checkpoint needed).
!python predictions_to_submission.py --predictions "{ROOT}/Outputs/predictions_maf_*.csv" --sample_submission "{SAMPLE_SUB}"

`{ROOT}/Outputs/submission.csv` is inside `/kaggle/working`, so it is kept automatically
as this notebook's Output when you commit - download it from the notebook's Output tab
and upload it on the competition's Submit Predictions page.